In [49]:
#prepare ordinary kriging rasters as predictor variable for random forest models
#perform leave-one-out kriging to prevent target leakage for model
#remove stations with strong correlations (> 0.70) & within 1 pixel distance of coarsest predictor raster (~16 km)

# 1. Identify valid stations for day d and remove those within 1 pixel distance (~1.4 km) 
# 2. Calculate global daily residuals for all valid stations (residual = actual observation - seasonal linear trend).
# 3. Nscore transform these global residuals to obtain a Gaussian distribution.
# 4. Krige these residuals using seasonal variogram parameters to create a "Global Gaussian Residual Raster" (1 km).
# 5. **Loop within station i (LOO Point Prediction)**
    # Determine select stations for point i (exclude station i and any stations too close/correlated).
    # Read precipitation residuals of these select stations.
    # Nscore transform these residuals (building a lookup table specifically excluding station i).
    # Perform **Point Kriging** at the exact coordinate of station i to get a single Gaussian LOO value.
    # **Replace** the pixel value at station i's location in the "Global Gaussian Residual Raster" with this LOO value.
# **End station loop i**

# 6. Reverse nscore transform the entire modified "Global Gaussian Residual Raster".
# 7. Add the seasonal linear trend surface (1 km raster) back to the reversed residual raster.
# 8. Clamp all negative values in the final raster to 0.
# 9. Save the final kriged raster for day d as the predictor variable for Random Forest.

In [50]:
import os
import numpy as np
import pandas as pd
import geostatspy.geostats as geostats
import geostatspy.GSLIB as GSLIB
import matplotlib.pyplot as plt
import gc
from pyproj import Proj
from pykrige.ok import OrdinaryKriging
import rasterio as rio
import xarray as xr
import rioxarray as rxr

In [51]:
# import all station observations and filter to include only training stations
all_prec = pd.read_csv(r'../Data/Processed/Insitu/insitu_rainfall_1991_2020.csv')
train_stations = pd.read_csv(r'../Data/Processed/Split_stations/train_stations.csv')
train_index = train_stations['index_no'].tolist()
stn_prec_df = all_prec[all_prec['index_no'].isin(train_index)].reset_index(drop=True)

#variogram parameters by season - range in km, a small nugget is kept in all seasons for kriging stability
variogram_params = {
    'winter': {'range': 17.3, 'sill': 0.695, 'nugget': 0.005},
    'pre_monsoon': {'range': 20.1, 'sill': 1.083, 'nugget': 0.005},
    'monsoon': {'range': 38.1, 'sill': 0.845, 'nugget': 0.121},
    'post_monsoon': {'range': 50.4, 'sill': 1.184, 'nugget': 0.122}
}

#spatial linear trend parameters by season - b1 = slope for lat, b2 = slope for long, c = intercept
trend_params = {
    'winter': {'b1': 25.7005, 'b2': 2.1057, 'c': -482.0265},
    'pre_monsoon': {'b1': 43.9990, 'b2': 35.0136, 'c': -3991.7142},
    'monsoon': {'b1': -79.2576, 'b2': 16.4877, 'c': 2026.6630},
    'post_monsoon': {'b1': -3.6445, 'b2': 3.7202, 'c': -156.1296}
}

seasons = {
    'winter': [12, 1, 2],
    'pre_monsoon': [3, 4, 5],
    'monsoon': [6, 7, 8, 9],
    'post_monsoon': [10, 11]
}

stn_prec_df['season'] = (stn_prec_df['month'].apply(lambda x: [k for k, v in seasons.items() if x in v][0]))
stn_prec_df['date'] = pd.to_datetime(stn_prec_df[['year', 'month', 'day']])
myProj = Proj("+proj=utm +zone=45 +ellps=WGS84 +datum=WGS84 +units=m +no_defs")
x, y = myProj(stn_prec_df['long'].values, stn_prec_df['lat'].values)
stn_prec_df['X_km'] = x / 1000.0
stn_prec_df['Y_km'] = y / 1000.0

stn_prec_df = stn_prec_df[['date', 'index_no', 'lat', 'long', 'X_km', 'Y_km', 'season', 'rainfall_sum']]

del all_prec, train_stations, train_index
gc.collect()

0

In [ ]:
# Helper functions 

def perform_kriging(x_array, y_array, z_array, v_range, nugget, sill, chunk_size=500):
    grid_resolution = 1.0  # 1 km
    x_min, x_max = x_array.min(), x_array.max()
    y_min, y_max = y_array.min(), y_array.max()

    # Create 1D axis arrays. PyKrige 'grid' style expects axis vectors, not a meshgrid.
    grid_x = np.arange(x_min, x_max + grid_resolution, grid_resolution)
    grid_y = np.arange(y_min, y_max + grid_resolution, grid_resolution)
    
    OK = OrdinaryKriging(
        x_array, y_array, z_array,
        variogram_model='gaussian',
        variogram_parameters={'range': v_range, 'sill': sill, 'nugget': nugget},
        enable_plotting=False,
        coordinates_type='geographic')

    # Execute kriging in chunks along the Y axis to avoid MemoryError
    z_interp_list = []
    # Using the built-in range() now that it isn't shadowed by 'v_range'
    for i in range(0, len(grid_y), chunk_size):
        chunk_y = grid_y[i:i + chunk_size]
        z_chunk, _ = OK.execute('grid', grid_x, chunk_y)
        z_interp_list.append(z_chunk)
    
    # Combined result
    z_interp = np.vstack(z_interp_list)
    return z_interp, grid_x, grid_y

def save_raster(value_array, grid_x, grid_y, output_path, filename):
    os.makedirs(output_path, exist_ok=True)
    # Create DataArray with proper spatial coordinates
    z_da = xr.DataArray(
        value_array,
        z_interp, 
        coords={'y': grid_y, 'x': grid_x}, 
        dims=['y', 'x'], 
        name="interpolated_value"
    )
    z_da.rio.write_crs("EPSG:4326", inplace=True) 
    z_da.rio.to_raster(os.path.join(output_path, filename))

In [53]:
# first we find stations within 1 pixel distance
unique_stations = stn_prec_df[['index_no', 'X_km', 'Y_km', 'lat', 'long']].drop_duplicates().reset_index(drop=True)

def calculate_euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

for i in range(len(unique_stations)):
    # Check if any other station is within 1 km
    current_station = unique_stations.iloc[i]
    for j in range(len(unique_stations)):
        if i != j:
            other_station = unique_stations.iloc[j]
            distance = calculate_euclidean_distance(
                current_station['X_km'], current_station['Y_km'],
                other_station['X_km'], other_station['Y_km']
            )
            if distance < 1.41:
                print(f"Station {current_station['index_no']} is within 1.4 km of station {other_station['index_no']}")

#no stations found within 1 pixel 

#prepare seasonal spatial trend df for detrending and retrending
spatial_trend_df = unique_stations.copy()
for season, params in trend_params.items():
    b1 = params['b1']
    b2 = params['b2']
    c = params['c']
    spatial_trend_df.loc[:, f'trend_{season}'] = (b1 * spatial_trend_df['lat'] +
                                                   b2 * spatial_trend_df['long'] + c)


In [ ]:
#main function/loop

start_day = stn_prec_df['date'].min()
end_day = stn_prec_df['date'].max()
date_range_list = pd.date_range(start=start_day, end=end_day, freq='D').to_list()
for days in date_range_list[0:3]:
    prec_day = stn_prec_df[stn_prec_df['date'] == days].drop(columns = 'date').reset_index(drop=True)
    if prec_day.empty:
        continue
        
    season = prec_day['season'].iloc[0]
    # Detrending
    merged_prec_trend_day = pd.merge(
        prec_day, spatial_trend_df, 
        on = ['index_no', 'lat', 'long', 'X_km', 'Y_km'], 
        how = 'inner'
    )
    merged_prec_trend_day['residual'] = merged_prec_trend_day['rainfall_sum'] - merged_prec_trend_day[f'trend_{season}']
    z_nscore, nscore_table = geostats.nscore(merged_prec_trend_day, 'residual')
    
    x = merged_prec_trend_day['long'].values
    y = merged_prec_trend_day['lat'].values
    z_interp, gx, gy = perform_kriging(
        x, y, z_nscore, 
        variogram_params[season]['range'], 
        variogram_params[season]['nugget'], 
        variogram_params[season]['sill'])
    z_residual = geostats.nback(z_interp, nscore_table, 'residual')
    
    

